# Sequential Covering Algorithm - Starter Notebook
This notebook demonstrates a simplified sequential covering approach that learns one rule at a time for a binary classification task.

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score

In [ ]:
data = pd.DataFrame(
    [
        {'outlook': 'sunny', 'humidity': 'high', 'windy': False, 'play': 'no'},
        {'outlook': 'sunny', 'humidity': 'normal', 'windy': False, 'play': 'yes'},
        {'outlook': 'overcast', 'humidity': 'high', 'windy': False, 'play': 'yes'},
        {'outlook': 'rain', 'humidity': 'normal', 'windy': False, 'play': 'yes'},
        {'outlook': 'rain', 'humidity': 'normal', 'windy': True, 'play': 'no'},
        {'outlook': 'overcast', 'humidity': 'normal', 'windy': True, 'play': 'yes'},
        {'outlook': 'sunny', 'humidity': 'normal', 'windy': True, 'play': 'yes'},
        {'outlook': 'rain', 'humidity': 'high', 'windy': False, 'play': 'yes'},
        {'outlook': 'sunny', 'humidity': 'high', 'windy': True, 'play': 'no'},
    ]
)
data

In [ ]:
def learn_rule(examples, target_col='play', positive_label='yes'):
    positive_examples = examples[examples[target_col] == positive_label]
    candidate_columns = [col for col in examples.columns if col != target_col]

    best_rule = None
    best_precision = -1.0
    best_coverage = -1

    for column in candidate_columns:
        for value in examples[column].unique():
            covered = examples[examples[column] == value]
            positives = (covered[target_col] == positive_label).sum()
            if len(covered) == 0:
                continue
            precision = positives / len(covered)
            coverage = positives
            if precision > best_precision or (precision == best_precision and coverage > best_coverage):
                best_precision = precision
                best_coverage = coverage
                best_rule = (column, value)

    return best_rule


remaining = data.copy()
rules = []
while (remaining['play'] == 'yes').any():
    column, value = learn_rule(remaining)
    rules.append((column, value))
    covered_positive_mask = (remaining[column] == value) & (remaining['play'] == 'yes')
    remaining = remaining.loc[~covered_positive_mask].copy()
    if remaining.empty:
        break

rules

In [ ]:
def predict_with_sequential_rules(row, learned_rules):
    for column, value in learned_rules:
        if row[column] == value:
            return 'yes'
    return 'no'


data['prediction'] = data.apply(lambda row: predict_with_sequential_rules(row, rules), axis=1)
accuracy = accuracy_score(data['play'], data['prediction'])
print('Learned rules:', rules)
print(f'Sequential covering accuracy: {accuracy:.3f}')
data